# 🔬 Transfer Learning — Pretrain .npy → Fine-tune CSV only
## Alternative à l'entraînement mixte : comparer les deux stratégies
### CMKL University · Stage 2026

---

**Question posée** : plutôt que d'entraîner sur .npy et CSV mélangés (approche déjà
testée, 94.62%/99.25%), que se passe-t-il si on part du modèle .npy déjà entraîné
et qu'on le fine-tune **uniquement** sur CSV, sans revoir de données .npy ?

**Avantage de cette approche** : ne nécessite PAS de refaire Phase 1 (SSL) ni
Phase 2 (Probing) — on repart directement du checkpoint final .npy déjà entraîné
(`final_model_v2.pth`, 95.25% test .npy) et on ne relance qu'un fine-tuning
court sur CSV. Beaucoup plus rapide que l'entraînement mixte complet.

**Risque à surveiller — catastrophic forgetting** : en ne voyant plus de données
.npy pendant le fine-tuning, le modèle pourrait "oublier" ce qu'il a appris sur
le domaine synthétique. C'est exactement ce que le modèle mixte a réussi à éviter
(-0.63 pts seulement sur .npy). On évalue donc sur **les deux test sets séparément**
pour mesurer précisément ce compromis.

**Comparaison à établir** :
```
                          Test .npy    Test CSV
.npy seul (référence)      95.25%       23.47%
Mixte (.npy+CSV, ensemble) 94.62%       99.25%
Transfer learning (ce nb)     ?            ?
```


---
## ⚙️ Section 0 — Imports & Configuration


In [ ]:
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
for pkg in ['scikit-learn', 'seaborn']:
    try: __import__(pkg.replace('-','_'))
    except ImportError: install(pkg)
print('✓ Packages prêts')

In [ ]:
import os, glob, re, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HOME   = os.path.expanduser('~')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

In [ ]:
from torch.utils.tensorboard import SummaryWriter

LOG_DIR = os.path.join(HOME, 'runs', 'patchtst_transfer')
os.makedirs(LOG_DIR, exist_ok=True)
writer  = SummaryWriter(LOG_DIR)
print(f'✓ TensorBoard logs → {LOG_DIR}')
print(f'  Terminal : tensorboard --logdir {LOG_DIR} --port 6006')

def safe_log(writer, *args, method='add_scalar', **kwargs):
    try: getattr(writer, method)(*args, **kwargs)
    except Exception: pass

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CONFIG — architecture identique au modèle .npy source (95.25%)
# ════════════════════════════════════════════════════════════════════
CFG = {
    'L'         : 6700,
    'WN_MIN'    : 650,
    'WN_MAX'    : 4000,
    'WN_STEP'   : 0.5,

    'patch_size' : 320,
    'stride'     : 256,

    'd_model'   : 256,
    'n_heads'   : 16,
    'n_layers'  : 5,
    'd_ff'      : 512,
    'dropout'   : 0.1,

    # ── Fine-tuning CSV-only : LR volontairement TRÈS petit ──────────
    # On ne veut PAS que le backbone dérive trop loin de ce qu'il a
    # appris sur .npy — sinon on perd l'intérêt du transfer learning
    # et on se rapproche d'un entraînement CSV from scratch.
    'alpha'      : 1.0,
    'beta'       : 0.2,     # beta bas — optimal identifié sur CSV seul
    'ft_epochs'  : 80,
    'ft_lr_backbone' : 5e-6,   # 2x PLUS petit que le FT .npy original (1e-5)
    'ft_lr_heads'    : 5e-5,   # 2x PLUS petit également
    'batch_size' : 16,        # plus petit dataset CSV → batch plus petit

    'npy_model_path'      : os.path.join(HOME, 'models', 'final_model_v2.pth'),
    'transfer_final_path' : os.path.join(HOME, 'models', 'final_model_transfer.pth'),
}

os.makedirs(os.path.join(HOME, 'models'), exist_ok=True)

WN_GRID   = np.arange(CFG['WN_MIN'], CFG['WN_MAX'], CFG['WN_STEP'])
CFG['L']  = len(WN_GRID)
L = CFG['L']
N_PATCHES = (L - CFG['patch_size']) // CFG['stride'] + 2
print(f'L = {L}, N_PATCHES = {N_PATCHES}')
assert CFG['d_model'] % CFG['n_heads'] == 0

In [ ]:
ASSUMED_CLASSES = [
    'ABS', 'ACRYLIC', 'CELLULOSE', 'CHITOSAN', 'ENR', 'EPDM', 'EVA', 'HDPE',
    'LDPE', 'NYLON', 'PBAT', 'PBS', 'PC', 'PEEK', 'PEI', 'PET',
    'PF THERMOPLASTIC', 'PF THERMOSET', 'PHB', 'PLA', 'PMMA', 'POM', 'PP',
    'PS', 'PTFE', 'PU', 'PVA', 'PVC', 'PVDF', 'SAN',
]
N_CLASSES = len(ASSUMED_CLASSES)
CFG['N_CLASSES'] = N_CLASSES

le = LabelEncoder()
le.fit(ASSUMED_CLASSES)
print(f'{N_CLASSES} classes')

---
## 📊 Section 1 — Données .npy (TEST uniquement — mesure du forgetting)

On ne charge PAS le train .npy — le principe du transfer learning est de
**ne plus entraîner** sur .npy. On garde uniquement le test officiel pour
vérifier après coup si le modèle a "oublié" ce domaine.


In [ ]:
NPY_ROOT  = os.path.join(HOME, 'data', '2026-FTIR-Preprocesed')
TEST_DIR  = os.path.join(NPY_ROOT, '1.2 TestSet - UptoY dB')
noise = 'Upto30SNR'

def npy_path(base_dir, filename):
    p = os.path.join(base_dir, filename)
    if not os.path.exists(p): print(f'  ✗ INTROUVABLE : {p}')
    return p

npy_test_clean = np.load(npy_path(TEST_DIR, 'TestGroundTruthSet_Pre.npy'))
npy_test_noisy = np.load(npy_path(TEST_DIR, f'TestNoisySet_{noise}_Pre.npy'))

N_PER_CLASS_NPY_TEST = npy_test_clean.shape[0] // N_CLASSES
npy_labels_test = np.repeat(np.arange(N_CLASSES), N_PER_CLASS_NPY_TEST)

print(f'✓ .npy Test (officiel, pour mesure du forgetting) : {npy_test_noisy.shape[0]} spectres')

---
## 📁 Section 2 — Données CSV (source du fine-tuning)


In [ ]:
CSV_ROOT = os.path.join(HOME, 'data', '2026-FirstDataSet', '2026 - Complete FTIR Dataset')

PATHS_CSV = {
    '2023_base' : os.path.join(CSV_ROOT, '2023 Dataset - 22 MP Types with 10 Clean and 60 Noisy'),
    '2025_ext'  : os.path.join(CSV_ROOT, '2025 Dataset 1 - Same 22 MP Types - Add 40 Spectra'),
    '2025_new'  : os.path.join(CSV_ROOT, '2025 Dataset 2 - New 9 MP Types - 50 Clean and 100 Noisy'),
}
for name, path in PATHS_CSV.items():
    status = '✓' if os.path.exists(path) else '✗ INTROUVABLE'
    print(f'  {status}  {name}')

In [ ]:
EXCLUDE_FILES = {'ref.csv', 'reference.csv', 'background.csv', 'bg.csv'}

def is_noisy_csv(filepath):
    p = str(filepath).lower()
    if any(k in p for k in ['noisy', '_sd', '-sd', 'sd_']): return True
    if any(k in p for k in ['clean', '_rm', '-rm', 'rm_']): return False
    return False

def extract_label_csv(filepath):
    name = Path(filepath).stem.upper()
    for pattern in ['_SD_', '_RM_', '_NOISY', '_CLEAN', 'PARTICLE', '-NOISY',
                    '-CLEAN', '_50', '_60', '_40', '_100', '_10', '_30',
                    ' SPECTRUMS', ' SPECTUMS', 'ADD_40', '-ADD_40']:
        name = name.replace(pattern, ' ')
    name = re.sub(r'\d+', '', name)
    name = re.sub(r'\bNEW\b|\bJAN\b|\bX\b', '', name)
    name = ' '.join(name.replace('_', ' ').replace('-', ' ').split())
    MAPPING = {
        'NYLON PARTICLE' : 'NYLON', 'PTEE' : 'PTFE', 'PTFE' : 'PTFE',
        'PF THERMOPLASTIC CLEAN' : 'PF THERMOPLASTIC',
        'PF THERMOSET CLEAN'     : 'PF THERMOSET',
    }
    if name in MAPPING: return MAPPING[name]
    if name in ASSUMED_CLASSES: return name
    for cls in ASSUMED_CLASSES:
        if cls in name or name in cls: return cls
    return None

def read_csv_multispectra(filepath, sep=','):
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        start_idx = 0
        for i, line in enumerate(lines):
            parts = line.strip().split(sep)
            if len(parts) >= 2:
                try:
                    float(parts[0].replace(',', '.'))
                    start_idx = i; break
                except ValueError: continue
        valid = ''.join(lines[start_idx:])
        headers = lines[start_idx-1].strip().split(sep) if start_idx > 0 else []
        try:
            df = pd.read_csv(io.StringIO(valid), sep=sep, header=None, decimal=',')
        except Exception:
            df = pd.read_csv(io.StringIO(valid), sep=sep, header=None, decimal='.')
        cols = []
        for ci, cn in enumerate(df.columns):
            h = headers[ci].upper() if ci < len(headers) else ''
            v = str(df[cn].iloc[0]).upper()
            if any(k in h for k in ['AIR','BACKGROUND','BG']): continue
            if any(k in v for k in ['AIR','BACKGROUND','BG']): continue
            cols.append(cn)
        df = df[cols].apply(pd.to_numeric, errors='coerce')
        df = df.dropna(subset=[df.columns[0]])
        if len(df) < 100: return None
        wn    = df.iloc[:, 0].values.astype(float)
        order = np.argsort(wn); wn = wn[order]
        spectra = []
        for c in range(1, df.shape[1]):
            ab = df.iloc[order, c].values.astype(float)
            if np.isnan(ab).all() or ab.std() < 1e-10: continue
            nans = np.isnan(ab)
            if nans.any():
                ab[nans] = np.interp(np.where(nans)[0], np.where(~nans)[0], ab[~nans])
            spectra.append(ab.astype(np.float32))
        return (wn, spectra) if spectra else None
    except Exception: return None

print('✓ Fonctions de lecture définies')

In [ ]:
print('Chargement des CSV (propres + bruités)...')
csv_records = []
for src_name, folder in PATHS_CSV.items():
    if not os.path.exists(folder): continue
    files = glob.glob(os.path.join(folder, '**/*.csv'), recursive=True)
    n_ok = 0
    for fp in files:
        if Path(fp).name.lower() in EXCLUDE_FILES: continue
        label = extract_label_csv(fp)
        if label is None: continue
        result = read_csv_multispectra(fp)
        if result is None: continue
        wn, spectra_list = result
        noisy = is_noisy_csv(fp)
        for sp in spectra_list:
            sp_interp = np.interp(WN_GRID, wn, sp).astype(np.float32)
            csv_records.append({'label': label, 'is_noisy': noisy, 'spectrum': sp_interp})
            n_ok += 1
    print(f'  ✓ {src_name:12s} : {n_ok:4d} spectres')

df_csv_all = pd.DataFrame(csv_records)
df_csv_all['label_enc'] = le.transform(df_csv_all['label'])
print(f'\n  Total CSV : {len(df_csv_all)} spectres '
      f'(propres={  (~df_csv_all.is_noisy).sum()}, bruités={df_csv_all.is_noisy.sum()})')

In [ ]:
# ── Cibles de denoising proxy par classe (identique au notebook mixte) ────
df_csv_clean = df_csv_all[~df_csv_all['is_noisy']].reset_index(drop=True)
df_csv_noisy = df_csv_all[ df_csv_all['is_noisy']].reset_index(drop=True)

csv_clean_reference = {}
for cls_idx in range(N_CLASSES):
    subset = df_csv_clean[df_csv_clean['label_enc'] == cls_idx]['spectrum']
    if len(subset) > 0:
        csv_clean_reference[cls_idx] = np.mean(np.stack(subset.values), axis=0).astype(np.float32)

csv_global_clean_mean = (np.mean(np.stack(df_csv_clean['spectrum'].values), axis=0).astype(np.float32)
                         if len(df_csv_clean) > 0 else np.zeros(L, dtype=np.float32))

print(f'Référence propre disponible pour {len(csv_clean_reference)}/{N_CLASSES} classes')

In [ ]:
# ── Split CSV : même logique que le notebook mixte (70/15/15) ─────────────
csv_noisy_counts = Counter(df_csv_noisy['label_enc'])
csv_singleton = {k for k, v in csv_noisy_counts.items() if v < 3}
df_csv_multi  = df_csv_noisy[~df_csv_noisy['label_enc'].isin(csv_singleton)]
df_csv_single = df_csv_noisy[ df_csv_noisy['label_enc'].isin(csv_singleton)]

idx_tr_csv, idx_valtest_csv = train_test_split(
    range(len(df_csv_multi)), test_size=0.3,
    random_state=SEED, stratify=df_csv_multi['label_enc'])
idx_val_csv, idx_test_csv = train_test_split(idx_valtest_csv, test_size=0.5, random_state=SEED)

df_csv_train = pd.concat([df_csv_multi.iloc[idx_tr_csv], df_csv_single]).reset_index(drop=True)
df_csv_val   = df_csv_multi.iloc[idx_val_csv].reset_index(drop=True)
df_csv_test  = df_csv_multi.iloc[idx_test_csv].reset_index(drop=True)

print(f'CSV Train : {len(df_csv_train)}   Val : {len(df_csv_val)}   Test : {len(df_csv_test)}')
print('(Même split que le notebook mixte — comparaison directe possible)')

In [ ]:
class SimpleMultiTaskDataset(Dataset):
    def __init__(self, noisy, clean, labels):
        self.noisy  = noisy.astype(np.float32)
        self.clean  = clean.astype(np.float32)
        self.labels = labels.astype(np.int64)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        x       = torch.tensor(self.noisy[idx], dtype=torch.float32)
        x_clean = torch.tensor(self.clean[idx], dtype=torch.float32)
        y       = torch.tensor(self.labels[idx], dtype=torch.long)
        mu, sigma = x.mean(), x.std() + 1e-8
        x       = (x - mu) / sigma
        x_clean = (x_clean - mu) / sigma
        return x, x_clean, y

def build_clean_targets(df):
    return np.stack([csv_clean_reference.get(int(l), csv_global_clean_mean)
                     for l in df['label_enc'].values])

csv_train_spectra = np.stack(df_csv_train['spectrum'].values)
csv_train_clean   = build_clean_targets(df_csv_train)
csv_val_spectra   = np.stack(df_csv_val['spectrum'].values)
csv_val_clean     = build_clean_targets(df_csv_val)
csv_test_spectra  = np.stack(df_csv_test['spectrum'].values)
csv_test_clean    = build_clean_targets(df_csv_test)

train_dataset = SimpleMultiTaskDataset(csv_train_spectra, csv_train_clean, df_csv_train['label_enc'].values)
val_dataset   = SimpleMultiTaskDataset(csv_val_spectra,   csv_val_clean,   df_csv_val['label_enc'].values)
test_csv_dataset = SimpleMultiTaskDataset(csv_test_spectra, csv_test_clean, df_csv_test['label_enc'].values)
test_npy_dataset = SimpleMultiTaskDataset(npy_test_noisy, npy_test_clean, npy_labels_test)

# ── Sampler pondéré par classe (déséquilibre CSV entre polymères) ─────────
train_class_counts = np.bincount(df_csv_train['label_enc'].values, minlength=N_CLASSES)
sample_weights = 1.0 / (train_class_counts[df_csv_train['label_enc'].values] + 1e-8)
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float32),
    num_samples=len(train_dataset), replacement=True)

train_loader    = DataLoader(train_dataset, batch_size=CFG['batch_size'],
                             sampler=sampler, num_workers=0)
val_loader      = DataLoader(val_dataset, batch_size=CFG['batch_size'],
                             shuffle=False, num_workers=0)
test_csv_loader = DataLoader(test_csv_dataset, batch_size=CFG['batch_size'],
                             shuffle=False, num_workers=0)
test_npy_loader = DataLoader(test_npy_dataset, batch_size=CFG['batch_size'],
                             shuffle=False, num_workers=0)

print(f'✓ DataLoaders prêts')
print(f'  train_dataset (CSV only) : {len(train_dataset)}')
print(f'  val_dataset   (CSV only) : {len(val_dataset)}')
print(f'  test_csv_dataset         : {len(test_csv_dataset)}')
print(f'  test_npy_dataset (forgetting check) : {len(test_npy_dataset)}')

---
## 🏛️ Section 3 — Architecture (identique au modèle .npy source)


In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, L, patch_size, stride, d_model):
        super().__init__()
        self.P, self.S, self.D = patch_size, stride, d_model
        self.N = (L - patch_size) // stride + 2
        self.patch_proj = nn.Linear(patch_size, d_model)
        self.pos_embed  = nn.Embedding(self.N, d_model)
        self.dropout    = nn.Dropout(0.1)
    def get_raw_patches(self, x):
        B = x.shape[0]
        pad = x[:, -1:].expand(B, self.S)
        x_pad = torch.cat([x, pad], dim=1)
        return x_pad.unfold(1, self.P, self.S)
    def forward(self, x):
        patches  = self.get_raw_patches(x)
        content  = self.patch_proj(patches)
        pos_vecs = self.pos_embed(torch.arange(self.N, device=x.device))
        return self.dropout(content + pos_vecs)

class ConformerFFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.W1, self.V, self.W2 = (nn.Linear(d_model, d_ff), nn.Linear(d_model, d_ff),
                                     nn.Linear(d_ff, d_model))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = self.norm(x)
        return self.drop(self.W2(F.silu(self.W1(x)) * self.V(x)))

class ConformerConvModule(nn.Module):
    def __init__(self, d_model, kernel_size=31, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.pw1  = nn.Conv1d(d_model, 2*d_model, 1)
        self.glu  = nn.GLU(dim=1)
        self.dw   = nn.Conv1d(d_model, d_model, kernel_size, padding=kernel_size//2, groups=d_model)
        self.bn   = nn.BatchNorm1d(d_model)
        self.act  = nn.SiLU()
        self.pw2  = nn.Conv1d(d_model, d_model, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        r = x
        x = self.norm(x).transpose(1,2)
        x = self.glu(self.pw1(x))
        x = self.act(self.bn(self.dw(x)))
        x = self.drop(self.pw2(x)).transpose(1,2)
        return r + x

class ConformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout, kernel_size=31):
        super().__init__()
        self.ffn1 = ConformerFFN(d_model, d_ff, dropout)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(d_model)
        self.conv = ConformerConvModule(d_model, kernel_size, dropout)
        self.ffn2 = ConformerFFN(d_model, d_ff, dropout)
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = x + 0.5 * self.ffn1(x)
        xn = self.attn_norm(x)
        x  = x + self.drop(self.attn(xn, xn, xn)[0])
        x  = self.conv(x)
        x  = x + 0.5 * self.ffn2(x)
        return self.norm(x)

class TransformerBackbone(nn.Module):
    def __init__(self, d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.layers = nn.ModuleList([
            ConformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        for l in self.layers: x = l(x)
        return self.norm(x)

class ClassificationHead(nn.Module):
    def __init__(self, d_model, n_classes, dropout=0.1, hidden_dim=None):
        super().__init__()
        if hidden_dim is None: hidden_dim = d_model // 2
        self.attn_pool = nn.Linear(d_model, 1)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, n_classes))
    def forward(self, z):
        w = F.softmax(self.attn_pool(z), dim=1)
        return self.head((w * z).sum(dim=1))

class DenoisingHead(nn.Module):
    def __init__(self, d_model, patch_size, n_patches, stride, spectrum_length):
        super().__init__()
        self.P, self.S, self.N, self.L = patch_size, stride, n_patches, spectrum_length
        self.proj = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
                                   nn.GELU(), nn.Linear(d_model, patch_size))
    def forward(self, z):
        B = z.shape[0]
        pr = self.proj(z)
        out = torch.zeros(B, self.L+self.S, device=z.device)
        cnt = torch.zeros(self.L+self.S, device=z.device)
        for k in range(self.N):
            s = k * self.S
            out[:, s:s+self.P] += pr[:, k, :]
            cnt[s:s+self.P]    += 1
        return (out / cnt.clamp(min=1))[:, :self.L]

class PatchTSTMultiTask(nn.Module):
    def __init__(self, patch_embed, backbone, class_head, denoise_head, alpha=1.0, beta=0.5):
        super().__init__()
        self.patch_embed, self.backbone = patch_embed, backbone
        self.class_head, self.denoise_head = class_head, denoise_head
        self.alpha, self.beta = alpha, beta
    def encode(self, x):
        return self.backbone(self.patch_embed(x))
    def classify(self, x):
        return self.class_head(self.encode(x))
    def forward(self, x, y=None, clean_target=None):
        z = self.encode(x)
        logits   = self.class_head(z)
        denoised = self.denoise_head(z)
        loss = None
        if y is not None and clean_target is not None:
            loss_clf = F.cross_entropy(logits, y, label_smoothing=0.1)
            loss_den = F.mse_loss(denoised, clean_target)
            loss = self.alpha * loss_clf + self.beta * loss_den
        return logits, denoised, loss

print('✓ Architecture définie')

In [ ]:
patch_embed  = PatchEmbedding(CFG['L'], CFG['patch_size'], CFG['stride'], CFG['d_model']).to(DEVICE)
backbone     = TransformerBackbone(CFG['d_model'], CFG['n_heads'], CFG['n_layers'],
                                    CFG['d_ff'], CFG['dropout']).to(DEVICE)
class_head   = ClassificationHead(CFG['d_model'], CFG['N_CLASSES'], dropout=0.1).to(DEVICE)
denoise_head = DenoisingHead(CFG['d_model'], CFG['patch_size'], N_PATCHES,
                              CFG['stride'], CFG['L']).to(DEVICE)

model = PatchTSTMultiTask(patch_embed, backbone, class_head, denoise_head,
                          alpha=CFG['alpha'], beta=CFG['beta']).to(DEVICE)

total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total paramètres : {total:,}')

---
## 📦 Section 4 — Charger le modèle .npy source (point de départ du transfer)


In [ ]:
ckpt_source = torch.load(CFG['npy_model_path'], map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt_source['model_state'])
print(f'✓ Modèle .npy source chargé — Val Acc origine : {ckpt_source["val_acc"]:.2%}')

# ── Vérification AVANT fine-tuning : baseline sur les deux test sets ──────
@torch.no_grad()
def evaluate_full(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, x_clean, y in loader:
            x = x.to(DEVICE)
            logits, _, _ = model(x)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(y.numpy())
    return accuracy_score(all_labels, all_preds)

acc_npy_before = evaluate_full(model, test_npy_loader)
acc_csv_before = evaluate_full(model, test_csv_loader)

print(f'\n=== AVANT fine-tuning (modèle .npy tel quel) ===')
print(f'  Test .npy : {acc_npy_before:.2%}  (référence attendue ≈ 95.25%)')
print(f'  Test CSV  : {acc_csv_before:.2%}  (référence attendue ≈ 23.47%, domain gap)')

---
## 🎯 Section 5 — Fine-tuning sur CSV uniquement (transfer learning)

Tout le réseau est dégelé, mais avec des learning rates volontairement très
petits (2x plus petits que le fine-tuning .npy original) pour limiter la
dérive et préserver autant que possible les représentations apprises sur .npy.


In [ ]:
for p in model.parameters(): p.requires_grad = True

ft_optimizer = AdamW([
    {'params': model.patch_embed.parameters(),  'lr': CFG['ft_lr_backbone']},
    {'params': model.backbone.parameters(),     'lr': CFG['ft_lr_backbone']},
    {'params': model.class_head.parameters(),   'lr': CFG['ft_lr_heads']},
    {'params': model.denoise_head.parameters(), 'lr': CFG['ft_lr_heads']},
], weight_decay=1e-4)
ft_scheduler = CosineAnnealingLR(ft_optimizer, T_max=CFG['ft_epochs'], eta_min=1e-7)

print(f'  LR backbone : {CFG["ft_lr_backbone"]:.0e}')
print(f'  LR têtes    : {CFG["ft_lr_heads"]:.0e}')

In [ ]:
def snr_db(clean, signal):
    noise_power  = ((signal - clean) ** 2).mean(dim=-1) + 1e-8
    signal_power = (clean ** 2).mean(dim=-1) + 1e-8
    return 10 * torch.log10(signal_power / noise_power)

def multitask_train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, x_clean, y in loader:
        x, x_clean, y = x.to(DEVICE), x_clean.to(DEVICE), y.to(DEVICE)
        logits, denoised, loss = model(x, y=y, clean_target=x_clean)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()*len(y); correct += (logits.argmax(1)==y).sum().item(); n += len(y)
    return total_loss/n, correct/n

@torch.no_grad()
def multitask_eval_epoch(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, x_clean, y in loader:
        x, x_clean, y = x.to(DEVICE), x_clean.to(DEVICE), y.to(DEVICE)
        logits, denoised, loss = model(x, y=y, clean_target=x_clean)
        total_loss += loss.item()*len(y); correct += (logits.argmax(1)==y).sum().item(); n += len(y)
    return total_loss/n, correct/n

print('✓ Fonctions définies')

In [ ]:
ft_history = {'train_acc':[], 'val_csv_acc':[], 'val_npy_acc':[]}
best_combined_acc = 0.0

print(f'=== Transfer Learning Fine-tuning ({CFG["ft_epochs"]} époques, CSV only) ===')
print(f'{"Époque":>7} | {"Tr.Acc":>7} | {"Val CSV":>8} | {"Test npy (forgetting)":>22}')
print('-'*55)

for epoch in range(1, CFG['ft_epochs']+1):
    tr_loss, tr_acc = multitask_train_epoch(model, train_loader, ft_optimizer)
    va_loss, va_acc = multitask_eval_epoch(model, val_loader)
    ft_scheduler.step()

    ft_history['train_acc'].append(tr_acc)
    ft_history['val_csv_acc'].append(va_acc)

    safe_log(writer, 'Transfer/Acc_CSV', {'Train': tr_acc, 'Val': va_acc}, epoch, method='add_scalars')

    # Vérifier le forgetting sur .npy toutes les 10 epochs (coûteux : 18000 spectres)
    if epoch % 10 == 0 or epoch == 1 or epoch == CFG['ft_epochs']:
        acc_npy_now = evaluate_full(model, test_npy_loader)
        ft_history['val_npy_acc'].append((epoch, acc_npy_now))
        safe_log(writer, 'Transfer/Acc_npy_forgetting', acc_npy_now, epoch)
        forgetting_str = f'{acc_npy_now:.2%}'
    else:
        forgetting_str = '—'

    if epoch % 5 == 0:
        torch.save({
            'model_state': model.state_dict(), 'epoch': epoch,
            'ft_history': ft_history, 'cfg': CFG,
        }, os.path.join(HOME, 'models', 'checkpoint_transfer_latest.pth'))

    combined = (va_acc + (ft_history['val_npy_acc'][-1][1] if ft_history['val_npy_acc'] else acc_npy_before)) / 2
    if combined > best_combined_acc:
        best_combined_acc = combined
        torch.save({'model_state': model.state_dict(), 'cfg': CFG,
                    'le_classes': le.classes_, 'val_acc': best_combined_acc},
                   CFG['transfer_final_path'])
        flag = ' ★'
    else:
        flag = ''

    if epoch % 5 == 0 or epoch == 1:
        print(f'{epoch:7d} | {tr_acc:6.2%} | {va_acc:7.2%} | {forgetting_str:>22}{flag}')

writer.flush()
print(f'\n✓ Meilleure moyenne (CSV+npy)/2 : {best_combined_acc:.2%}')
print(f'  Sauvegardé → {CFG["transfer_final_path"]}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(ft_history['train_acc'], label='Train (CSV)')
axes[0].plot(ft_history['val_csv_acc'], label='Val (CSV)')
axes[0].set_title('Accuracy sur CSV pendant le fine-tuning', fontweight='bold')
axes[0].set_xlabel('Époque'); axes[0].legend(); axes[0].grid(alpha=0.3)

npy_epochs = [e for e, a in ft_history['val_npy_acc']]
npy_accs   = [a for e, a in ft_history['val_npy_acc']]
axes[1].plot(npy_epochs, npy_accs, marker='o', color='coral')
axes[1].axhline(0.9525, color='steelblue', linestyle='--', label='Référence .npy seul (95.25%)')
axes[1].set_title('Catastrophic forgetting — Test .npy pendant le fine-tuning CSV',
                  fontweight='bold')
axes[1].set_xlabel('Époque'); axes[1].set_ylabel('Accuracy .npy')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 📊 Section 6 — Évaluation finale et comparaison des 3 stratégies


In [ ]:
ckpt_final = torch.load(CFG['transfer_final_path'], map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt_final['model_state'])
model.eval()

acc_npy_final = evaluate_full(model, test_npy_loader)
acc_csv_final = evaluate_full(model, test_csv_loader)

print('═'*65)
print('  ÉVALUATION FINALE — TRANSFER LEARNING (.npy → CSV)')
print('═'*65)
print(f'  Test .npy (forgetting check) : {acc_npy_final:.2%}')
print(f'  Test CSV (objectif transfer) : {acc_csv_final:.2%}')
print('═'*65)

In [ ]:
# ── Comparaison des 3 stratégies ───────────────────────────────────────────
print('═'*75)
print('  COMPARAISON DES 3 STRATÉGIES')
print('═'*75)
print(f'{"Stratégie":35s} | {"Test .npy":>10} | {"Test CSV":>10}')
print('-'*75)
print(f'{".npy seul (référence)":35s} | {"95.25%":>10} | {"23.47%":>10}')
print(f'{"Mixte (.npy+CSV ensemble)":35s} | {"94.62%":>10} | {"99.25%":>10}')
print(f'{"Transfer learning (.npy→CSV)":35s} | {acc_npy_final:>9.2%} | {acc_csv_final:>9.2%}')
print('═'*75)

fig, ax = plt.subplots(figsize=(11, 6))
strategies = ['.npy seul\n(référence)', 'Mixte\n(.npy+CSV)', 'Transfer\nlearning']
x = np.arange(len(strategies))
width = 0.35

npy_vals = [0.9525, 0.9462, acc_npy_final]
csv_vals = [0.2347, 0.9925, acc_csv_final]

bars1 = ax.bar(x - width/2, [v*100 for v in npy_vals], width, label='Test .npy', color='#2E75B6')
bars2 = ax.bar(x + width/2, [v*100 for v in csv_vals], width, label='Test CSV', color='#ED7D31')
ax.bar_label(bars1, fmt='%.1f%%', padding=3, fontsize=10, fontweight='bold')
ax.bar_label(bars2, fmt='%.1f%%', padding=3, fontsize=10, fontweight='bold')

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Comparaison des stratégies de généralisation domaine',
             fontweight='bold', fontsize=13)
ax.set_xticks(x); ax.set_xticklabels(strategies)
ax.set_ylim(0, 110)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(HOME, 'comparison_3_strategies.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('=== Rapport de classification — Test CSV (transfer learning) ===')
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x, x_clean, y in test_csv_loader:
        logits, _, _ = model(x.to(DEVICE))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(y.numpy())

classes_present = sorted(set(all_labels))
names_present = [le.classes_[i] for i in classes_present]
print(classification_report(all_labels, all_preds, labels=classes_present,
                             target_names=names_present, zero_division=0))

In [ ]:
# ── Conclusion ──────────────────────────────────────────────────────────
gap_transfer = acc_npy_final - acc_csv_final
gap_mixte = 0.9462 - 0.9925

print('═'*65)
print('  CONCLUSION')
print('═'*65)
if acc_npy_final < 0.85:
    print('  ⚠️  Catastrophic forgetting détecté : le modèle a perdu')
    print(f'     {0.9525 - acc_npy_final:.1%} de performance sur .npy en ne voyant plus ces données.')
else:
    print('  ✓  Pas de catastrophic forgetting majeur détecté.')

if acc_csv_final > 0.90:
    print(f'  ✓  Transfer réussi sur CSV : {acc_csv_final:.2%}')
else:
    print(f'  ⚠️  Transfer partiel sur CSV : {acc_csv_final:.2%} (moins bon que le mixte)')

print()
if (acc_npy_final + acc_csv_final)/2 > (0.9462 + 0.9925)/2:
    print('  → Transfer learning SURPASSE le modèle mixte en moyenne.')
else:
    print('  → Le modèle MIXTE reste la meilleure stratégie globale.')
print('═'*65)